# 03 — Raw Time-Series Explorer (P2)

Temporal inspection of raw metrics — complements `02_raw_distribution_explorer.ipynb`
(shape) with **when in the session** a metric's values occur, and whether that pattern
is consistent within a participant, across a participant's repeated sessions, or across
people entirely.

**Adapted from P1's `03_raw_time_series_explorer.ipynb`, not a blind port.** Two real
differences worth stating up front:

- **Panel structure matches P1's original design exactly**: one panel per device family
  (`mobile` / `desktop`), with every participant overlaid as a distinct-coloured line on
  the *same* axes within each panel — not split into one panel per participant (that's
  what `02`'s participant-split section already does; this notebook's job is temporal
  structure, and P1's device-panel-with-participant-overlay layout is the right way to
  see that against the same device-regime lens established in P1 and re-confirmed on
  P2's own first laptop session).
- **Time reference is session-relative, not phase-relative.** P1 zeroed `t_rel_s` to the
  start of whichever of its two fixed phases (typing / tapping) an event belonged to —
  reasonable for a task with exactly two back-to-back phases. P2 has 26 discrete task
  instances per session; zeroing per-task would produce a fragmented, near-useless
  trajectory. Here `t_rel_s = tRelMs / 1000`, relative to session start throughout —
  the same convention already used everywhere else in this project's analysis.
- `observed_session_number` uses `sessionIndex` directly — P1's own `infer_session_order`
  already treats `sessionIndex` as its second-priority fallback (after a dedicated
  `session_order` column P2 doesn't have), so this isn't a shortcut, it's the same
  precedent applied directly.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)

RAW_EVENTS_PARQUET = "../../data/processed/raw_events.parquet"
RAW_EVENTS_CSV = "../../data/processed/raw_events.csv"

raw = pd.read_parquet(RAW_EVENTS_PARQUET) if Path(RAW_EVENTS_PARQUET).exists() else pd.read_csv(RAW_EVENTS_CSV, low_memory=False)
raw['deviceFamily'] = raw['deviceFamily'].astype(str).str.lower()
print(f"{len(raw):,} raw events, {raw['sessionId'].nunique()} sessions, {raw['participantId'].nunique()} participants")
print("Device families present:", sorted(raw['deviceFamily'].unique()))

44,155 raw events, 4 sessions, 3 participants
Device families present: ['desktop', 'mobile']


## Raw metric registry

Same registry as `02_raw_distribution_explorer.ipynb`, duplicated here rather than
imported — matches P1's own precedent (both of P1's explorer notebooks keep an
independent copy of `RAW_FEATURE_FAMILIES` so each notebook runs standalone). The
extraction logic differs from `02`'s in one respect: every row here also carries
`t_rel_s`, needed for time-binning, which `02` doesn't need and deliberately doesn't
compute.

In [3]:
RAW_METRIC_REGISTRY = [
    ("typing", "typing_press_to_press_ms", "diff_consecutive", "keydown", None),
    ("typing", "typing_dwell_ms", "paired_duration", ("keydown", "keyup"), None),
    ("typing", "typing_value_length", "direct", "input", "payload_valueLength"),
    ("typing", "typing_delta_length", "direct", "input", "payload_deltaLength"),

    ("touch", "touch_hold_ms", "paired_duration", ("touchstart", "touchend"), None),
    ("touch", "touch_speed", "speed", "touchmove", None),
    ("touch", "touch_force", "direct", "touchstart", "payload_force"),
    ("touch", "touch_radiusX", "direct", "touchmove", "payload_radiusX"),
    ("touch", "touch_radiusY", "direct", "touchmove", "payload_radiusY"),

    ("pointer", "pointer_hold_ms", "paired_duration", ("pointerdown", "pointerup"), None),
    ("pointer", "pointer_speed", "speed", "pointermove", None),
    ("pointer", "pointer_pressure", "direct", "pointermove", "payload_pressure"),
    ("pointer", "pointer_width", "direct", "pointermove", "payload_width"),
    ("pointer", "pointer_height", "direct", "pointermove", "payload_height"),
    ("pointer", "pointer_tiltX", "direct", "pointermove", "payload_tiltX"),
    ("pointer", "pointer_tiltY", "direct", "pointermove", "payload_tiltY"),

    ("scroll", "scroll_velocity", "scroll_velocity", "scroll", None),

    ("motion", "motion_ax", "direct", "devicemotion", "payload_ax"),
    ("motion", "motion_ay", "direct", "devicemotion", "payload_ay"),
    ("motion", "motion_az", "direct", "devicemotion", "payload_az"),
    ("motion", "motion_agx", "direct", "devicemotion", "payload_agx"),
    ("motion", "motion_agy", "direct", "devicemotion", "payload_agy"),
    ("motion", "motion_agz", "direct", "devicemotion", "payload_agz"),
    ("motion", "motion_rotAlpha", "direct", "devicemotion", "payload_rotAlpha"),
    ("motion", "motion_rotBeta", "direct", "devicemotion", "payload_rotBeta"),
    ("motion", "motion_rotGamma", "direct", "devicemotion", "payload_rotGamma"),
    ("motion", "motion_magnitude", "magnitude", "devicemotion", ("payload_ax", "payload_ay", "payload_az")),

    ("orientation", "orientation_beta", "direct", "deviceorientation", "payload_beta"),
    ("orientation", "orientation_gamma", "direct", "deviceorientation", "payload_gamma"),
    ("orientation", "orientation_alpha_sin", "alpha_trig_sin", "deviceorientation", "payload_alpha"),
    ("orientation", "orientation_alpha_cos", "alpha_trig_cos", "deviceorientation", "payload_alpha"),

    ("gesture", "drag_distance", "direct", "gesture_drag_end", "payload_distancePx"),
    ("gesture", "drag_duration", "direct", "gesture_drag_end", "payload_durationMs"),
    ("gesture", "pot_drag_duration", "direct", "pot_drag_release", "payload_durationMs"),
    ("gesture", "approval_swipe_duration", "direct", "approval_swipe_release", "payload_durationMs"),
    ("gesture", "approval_swipe_ratio", "direct", "approval_swipe_release", "payload_swipeRatio"),
]

FAMILIES = sorted(set(r[0] for r in RAW_METRIC_REGISTRY))
print("Families:", FAMILIES)

Families: ['gesture', 'motion', 'orientation', 'pointer', 'scroll', 'touch', 'typing']


## Extraction functions — time-preserving version

In [4]:
def _meta_cols(df):
    return df[['participantId', 'sessionId', 'deviceFamily', 'sessionIndex']].reset_index(drop=True)


def _paired_duration_per_session(sub, start_kind, end_kind):
    rows = []
    for sid, s in sub.groupby('sessionId'):
        starts = s.loc[s['kind'] == start_kind, ['tRelMs', 'participantId', 'deviceFamily', 'sessionIndex']].sort_values('tRelMs')
        ends = s.loc[s['kind'] == end_kind, 'tRelMs'].sort_values()
        start_times = starts['tRelMs'].tolist()
        used = [False] * len(start_times)
        for end_t in ends:
            best_i, best_dt = None, None
            for i, s_t in enumerate(start_times):
                if used[i] or s_t > end_t:
                    continue
                dt = end_t - s_t
                if best_dt is None or dt < best_dt:
                    best_dt, best_i = dt, i
            if best_i is not None:
                used[best_i] = True
                row = starts.iloc[best_i]
                rows.append({'sessionId': sid, 'participantId': row['participantId'],
                             'deviceFamily': row['deviceFamily'], 'sessionIndex': row['sessionIndex'],
                             'value': best_dt, 't_rel_s': end_t / 1000.0})
    return pd.DataFrame(rows)


def _diff_consecutive_per_session(sub, kind):
    rows = []
    for sid, s in sub.groupby('sessionId'):
        k = s.loc[s['kind'] == kind].sort_values('tRelMs')
        if len(k) < 2:
            continue
        diffs = k['tRelMs'].diff().dropna()
        meta = k.iloc[1:][['participantId', 'deviceFamily', 'sessionIndex']].reset_index(drop=True)
        out = pd.DataFrame({'value': diffs.values, 't_rel_s': (k.iloc[1:]['tRelMs'] / 1000.0).values})
        out = pd.concat([out.reset_index(drop=True), meta], axis=1)
        out['sessionId'] = sid
        rows.append(out)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId'])


def _speed_per_session(sub, kind):
    rows = []
    for sid, s in sub.groupby('sessionId'):
        k = s.loc[s['kind'] == kind].sort_values('tRelMs')
        if len(k) < 2 or not {'payload_x', 'payload_y'}.issubset(k.columns):
            continue
        x = pd.to_numeric(k['payload_x'], errors='coerce').to_numpy(dtype=float)
        y = pd.to_numeric(k['payload_y'], errors='coerce').to_numpy(dtype=float)
        t = pd.to_numeric(k['tRelMs'], errors='coerce').to_numpy(dtype=float)
        valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(t)
        x, y, t = x[valid], y[valid], t[valid]
        if len(x) < 2:
            continue
        dt = np.diff(t) / 1000.0
        dist = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
        with np.errstate(divide='ignore', invalid='ignore'):
            speed = dist / dt
        meta = k.iloc[np.where(valid)[0][1:]][['participantId', 'deviceFamily', 'sessionIndex']].reset_index(drop=True)
        n = min(len(speed), len(meta))
        out = pd.DataFrame({'value': speed[:n], 't_rel_s': t[1:][:n] / 1000.0})
        out = pd.concat([out.reset_index(drop=True), meta.iloc[:n].reset_index(drop=True)], axis=1)
        out['sessionId'] = sid
        rows.append(out)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId'])


def _scroll_velocity_per_session(sub):
    rows = []
    for sid, s in sub.groupby('sessionId'):
        k = s.loc[s['kind'] == 'scroll'].sort_values('tRelMs')
        if len(k) < 2 or 'payload_scrollTop' not in k.columns:
            continue
        t = pd.to_numeric(k['tRelMs'], errors='coerce')
        top = pd.to_numeric(k['payload_scrollTop'], errors='coerce')
        dt = (t.diff() / 1000.0).replace(0, np.nan)
        velocity = (top.diff() / dt).replace([np.inf, -np.inf], np.nan)
        meta = k[['participantId', 'deviceFamily', 'sessionIndex']].reset_index(drop=True)
        out = pd.DataFrame({'value': velocity.values, 't_rel_s': (t / 1000.0).values})
        out = pd.concat([out.reset_index(drop=True), meta], axis=1)
        out['sessionId'] = sid
        rows.append(out.dropna(subset=['value']))
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId'])


def build_raw_metric_df(raw_df, registry):
    frames = []
    for family, metric, ext_type, kind, field in registry:
        if ext_type == "direct":
            sub = raw_df.loc[raw_df['kind'] == kind].copy()
            if field not in sub.columns:
                continue
            sub['value'] = pd.to_numeric(sub[field], errors='coerce')
            sub['t_rel_s'] = sub['tRelMs'] / 1000.0
            sub = sub.dropna(subset=['value'])
            out = sub[['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId']].copy()
        elif ext_type == "diff_consecutive":
            out = _diff_consecutive_per_session(raw_df, kind)
        elif ext_type == "paired_duration":
            out = _paired_duration_per_session(raw_df, kind[0], kind[1])
        elif ext_type == "speed":
            out = _speed_per_session(raw_df, kind)
        elif ext_type == "scroll_velocity":
            out = _scroll_velocity_per_session(raw_df)
        elif ext_type == "magnitude":
            sub = raw_df.loc[raw_df['kind'] == kind].copy()
            if not all(f in sub.columns for f in field):
                continue
            vals = [pd.to_numeric(sub[f], errors='coerce') for f in field]
            sub['value'] = np.sqrt(sum(v ** 2 for v in vals))
            sub['t_rel_s'] = sub['tRelMs'] / 1000.0
            sub = sub.dropna(subset=['value'])
            out = sub[['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId']].copy()
        elif ext_type in ("alpha_trig_sin", "alpha_trig_cos"):
            sub = raw_df.loc[raw_df['kind'] == kind].copy()
            if field not in sub.columns:
                continue
            rad = np.radians(pd.to_numeric(sub[field], errors='coerce'))
            sub['value'] = np.sin(rad) if ext_type == "alpha_trig_sin" else np.cos(rad)
            sub['t_rel_s'] = sub['tRelMs'] / 1000.0
            sub = sub.dropna(subset=['value'])
            out = sub[['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId']].copy()
        else:
            continue

        if out.empty:
            continue
        out['family'] = family
        out['metric'] = metric
        out['observed_session_number'] = pd.to_numeric(out['sessionIndex'], errors='coerce')
        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=['value', 't_rel_s', 'participantId', 'deviceFamily', 'sessionIndex', 'sessionId', 'family', 'metric', 'observed_session_number'])
    return pd.concat(frames, ignore_index=True)


raw_metric_df = build_raw_metric_df(raw, RAW_METRIC_REGISTRY)
print(f"{len(raw_metric_df):,} time-referenced metric observations across {raw_metric_df['metric'].nunique()} metrics")
raw_metric_df.groupby(['family', 'metric']).size().rename('n_obs').reset_index()

198,206 time-referenced metric observations across 36 metrics


,family,metric,n_obs
0,gesture,approval_swipe_duration,12
1,gesture,approval_swipe_ratio,12
2,gesture,drag_distance,21
3,gesture,drag_duration,21
4,gesture,pot_drag_duration,9
5,motion,motion_agx,11122
6,motion,motion_agy,11122
7,motion,motion_agz,11122
8,motion,motion_ax,11122
9,motion,motion_ay,11122


## Helper functions — ported directly from P1, column names adapted (`device_family` -> `deviceFamily`, `t_rel_s` unchanged)

In [5]:
def filter_time_range(df, time_range_s=(0.0, 400.0)):
    lo, hi = time_range_s
    out = df.loc[df['t_rel_s'].notna()].copy()
    return out.loc[(out['t_rel_s'] >= lo) & (out['t_rel_s'] <= hi)].copy()


def assign_time_bins(df, bin_seconds=5.0):
    out = df.copy()
    out['time_bin_left'] = np.floor(out['t_rel_s'] / bin_seconds) * bin_seconds
    out['time_bin_mid'] = out['time_bin_left'] + (bin_seconds / 2.0)
    return out


def get_participants_to_show(df, max_participants=None):
    counts = df.groupby('participantId', dropna=False)['sessionId'].nunique().sort_values(ascending=False)
    participants = counts.index.astype(str).tolist()
    return participants[:max_participants] if max_participants else participants


def make_participant_color_map(participants):
    cmap = plt.get_cmap('tab10')
    return {p: cmap(i % 10) for i, p in enumerate(participants)}


def device_panels_present(df, device_col='deviceFamily'):
    if device_col not in df.columns:
        return [('all', df.copy())]
    families = []
    for fam in ['desktop', 'mobile']:
        sub = df.loc[df[device_col].astype(str).str.lower() == fam].copy()
        if not sub.empty:
            families.append((fam, sub))
    return families if families else [('all', df.copy())]


def robust_display_limits(values, lower_q=0.01, upper_q=0.99, pad_frac=0.06):
    s = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if s.empty:
        return None
    lo, hi = float(np.nanquantile(s, lower_q)), float(np.nanquantile(s, upper_q))
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if hi <= lo:
        lo, hi = float(s.min()), float(s.max())
    if hi <= lo:
        pad = max(abs(lo) * pad_frac, 1.0)
        return lo - pad, hi + pad
    pad = (hi - lo) * pad_frac
    return lo - pad, hi + pad


def apply_robust_ylim(ax, values, lower_q=0.0, upper_q=1.0, annotate=False, pad_frac=0.12):
    lims = robust_display_limits(values, lower_q, upper_q, pad_frac=pad_frac)
    if lims is None:
        return
    ax.set_ylim(*lims)
    if annotate:
        ax.text(0.995, 0.98, f"display clipped to {int(lower_q*100)}-{int(upper_q*100)}th pct",
                 transform=ax.transAxes, ha='right', va='top', fontsize=9, alpha=0.75)

## Within-session trajectories

**This is the P1 layout you showed** — one panel per device family, every participant
overlaid in their own colour on the same axes. Faint lines are individual
session-level binned trajectories; the bold line + shaded ribbon is each participant's
median and IQR across all their sessions in that device panel.

In [6]:
WITHIN_SESSION_LINE_ALPHA = 0.25
WITHIN_SESSION_LINEWIDTH = 1.0
PROFILE_RIBBON_ALPHA = 0.18
TIME_RANGE_S = (0.0, 400.0)
BIN_SECONDS = 5.0


def plot_within_session_spaghetti_stacked(df, metric, bin_seconds=BIN_SECONDS, time_range_s=TIME_RANGE_S,
                                            min_points_per_session=3, device_col='deviceFamily', max_participants=None):
    sub = df.loc[df['metric'] == metric].copy()
    sub = filter_time_range(sub, time_range_s)
    if sub.empty:
        print(f"No time-aligned data for {metric}")
        return

    participants = get_participants_to_show(sub, max_participants)
    sub = sub.loc[sub['participantId'].isin(participants)].copy()
    sub = assign_time_bins(sub, bin_seconds)
    cmap = make_participant_color_map(participants)
    panels = device_panels_present(sub, device_col)

    fig, axes = plt.subplots(len(panels), 1, figsize=(18, 4.6 * len(panels)), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, (fam, fam_df) in zip(axes, panels):
        grouped = (fam_df.groupby(['participantId', 'sessionId', 'time_bin_mid'], dropna=False)
                   .agg(value_median=('value', 'median'), n=('value', 'size')).reset_index())
        session_sizes = grouped.groupby(['participantId', 'sessionId'], dropna=False)['n'].sum().reset_index(name='n_points')
        keep = session_sizes.loc[session_sizes['n_points'] >= min_points_per_session, ['participantId', 'sessionId']]
        grouped = grouped.merge(keep, on=['participantId', 'sessionId'], how='inner')

        if grouped.empty:
            ax.set_title(f"Within-session trajectories: {metric} ({fam}) — no sessions after support filter")
            ax.axis('off')
            continue

        for (participant, session_id), s in grouped.groupby(['participantId', 'sessionId'], dropna=False):
            s = s.sort_values('time_bin_mid')
            ax.plot(s['time_bin_mid'], s['value_median'], color=cmap.get(str(participant), 'C0'),
                     alpha=WITHIN_SESSION_LINE_ALPHA, linewidth=WITHIN_SESSION_LINEWIDTH, zorder=1)

        profile = (grouped.groupby(['participantId', 'time_bin_mid'], dropna=False)
                   .agg(traj_median=('value_median', 'median'),
                        traj_q25=('value_median', lambda x: np.nanquantile(x, 0.25)),
                        traj_q75=('value_median', lambda x: np.nanquantile(x, 0.75))).reset_index())

        for participant, p in profile.groupby('participantId', dropna=False):
            p = p.sort_values('time_bin_mid')
            color = cmap.get(str(participant), 'C0')
            ax.fill_between(p['time_bin_mid'], p['traj_q25'], p['traj_q75'], color=color, alpha=PROFILE_RIBBON_ALPHA, zorder=2)
            ax.plot(p['time_bin_mid'], p['traj_median'], color=color, linewidth=2.2, label=str(participant), zorder=3)

        ax.set_title(f"Within-session raw-behaviour trajectories: {metric} ({fam})")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.25)
        plotted_values = pd.concat([grouped['value_median'], profile['traj_q25'], profile['traj_q75']])
        apply_robust_ylim(ax, plotted_values)
        ax.legend(title="Participant", bbox_to_anchor=(1.01, 1), loc='upper left')

    axes[-1].set_xlabel("Session-relative time (s)")
    plt.tight_layout()
    plt.show()

## Participant-average temporal profiles — same device-panel-overlay layout, without the faint per-session spaghetti

In [7]:
def plot_participant_average_profiles_stacked(df, metric, bin_seconds=BIN_SECONDS, time_range_s=TIME_RANGE_S,
                                                device_col='deviceFamily', max_participants=None):
    sub = df.loc[df['metric'] == metric].copy()
    sub = filter_time_range(sub, time_range_s)
    if sub.empty:
        print(f"No time-aligned data for {metric}")
        return

    participants = get_participants_to_show(sub, max_participants)
    sub = sub.loc[sub['participantId'].isin(participants)].copy()
    sub = assign_time_bins(sub, bin_seconds)
    cmap = make_participant_color_map(participants)
    panels = device_panels_present(sub, device_col)

    fig, axes = plt.subplots(len(panels), 1, figsize=(18, 4.6 * len(panels)), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, (fam, fam_df) in zip(axes, panels):
        profile = (fam_df.groupby(['participantId', 'time_bin_mid'], dropna=False)
                   .agg(value_median=('value', 'median'),
                        value_q25=('value', lambda x: np.nanquantile(x, 0.25)),
                        value_q75=('value', lambda x: np.nanquantile(x, 0.75))).reset_index())
        if profile.empty:
            ax.set_title(f"Participant-average temporal profiles: {metric} ({fam}) — no data")
            ax.axis('off')
            continue

        for participant, p in profile.groupby('participantId', dropna=False):
            p = p.sort_values('time_bin_mid')
            color = cmap.get(str(participant), 'C0')
            ax.fill_between(p['time_bin_mid'], p['value_q25'], p['value_q75'], color=color, alpha=PROFILE_RIBBON_ALPHA, zorder=1)
            ax.plot(p['time_bin_mid'], p['value_median'], color=color, linewidth=2.3, label=str(participant), zorder=2)

        ax.set_title(f"Participant-average temporal profile: {metric} ({fam})")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.25)
        plotted_values = pd.concat([profile['value_median'], profile['value_q25'], profile['value_q75']])
        apply_robust_ylim(ax, plotted_values)
        ax.legend(title="Participant", bbox_to_anchor=(1.01, 1), loc='upper left')

    axes[-1].set_xlabel("Session-relative time (s)")
    plt.tight_layout()
    plt.show()

## Across-session trajectories

Session-median-of-metric plotted against `observed_session_number` (from `sessionIndex`)
— the actual tool for behavioural-drift tracking. Only meaningful for participants with
multiple sessions; with the current cohort that's `pA6XL23` alone, but this is
infrastructure meant to get more useful as the cohort grows, not a one-off check.

In [8]:
def plot_session_trajectory_by_device_stacked(df, metric, time_range_s=TIME_RANGE_S, device_col='deviceFamily', max_participants=None):
    sub = df.loc[df['metric'] == metric].copy()
    sub = filter_time_range(sub, time_range_s)
    if sub.empty:
        print(f"No time-aligned data for {metric}")
        return

    participants = get_participants_to_show(sub, max_participants)
    sub = sub.loc[sub['participantId'].isin(participants)].copy()
    cmap = make_participant_color_map(participants)

    session_summary = (sub.groupby(['participantId', 'sessionId', 'observed_session_number', device_col], dropna=False)
                        .agg(session_median=('value', 'median'), n_obs=('value', 'size')).reset_index()
                        .sort_values(['participantId', 'observed_session_number', 'sessionId']))

    panels = device_panels_present(session_summary, device_col)
    fig, axes = plt.subplots(len(panels), 1, figsize=(18, 4.4 * len(panels)), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, (fam, fam_df) in zip(axes, panels):
        if fam_df.empty:
            ax.set_title(f"Session trajectory: session median {metric} ({fam}) — no data")
            ax.axis('off')
            continue
        for participant, p in fam_df.groupby('participantId', dropna=False):
            p = p.sort_values('observed_session_number')
            ax.plot(p['observed_session_number'], p['session_median'], marker='o', linewidth=2.0,
                     markersize=5, color=cmap.get(str(participant), 'C0'), label=str(participant))
        ax.set_title(f"Session trajectory: session median {metric} ({fam})")
        ax.set_ylabel(f"Session median {metric}")
        ax.grid(True, alpha=0.25)
        apply_robust_ylim(ax, fam_df['session_median'])
        ax.legend(title="Participant", bbox_to_anchor=(1.01, 1), loc='upper left')

    axes[-1].set_xlabel("Observed session number")
    plt.tight_layout()
    plt.show()

## Run it — one family at a time

In [ ]:
FEATURE_FAMILY = "all"   # change to any family in FAMILIES, or "all"
SELECTED = FAMILIES if FEATURE_FAMILY == "all" else [FEATURE_FAMILY]
metrics_in_scope = raw_metric_df.loc[raw_metric_df['family'].isin(SELECTED), 'metric'].unique().tolist()
print("Metrics in scope:", metrics_in_scope)

for m in metrics_in_scope:
    plot_within_session_spaghetti_stacked(raw_metric_df, m)

Metrics in scope: ['typing_press_to_press_ms', 'typing_dwell_ms', 'typing_value_length', 'typing_delta_length', 'touch_hold_ms', 'touch_speed', 'touch_force', 'touch_radiusX', 'touch_radiusY', 'pointer_hold_ms', 'pointer_speed', 'pointer_pressure', 'pointer_width', 'pointer_height', 'pointer_tiltX', 'pointer_tiltY', 'scroll_velocity', 'motion_ax', 'motion_ay', 'motion_az', 'motion_agx', 'motion_agy', 'motion_agz', 'motion_rotAlpha', 'motion_rotBeta', 'motion_rotGamma', 'motion_magnitude', 'orientation_beta', 'orientation_gamma', 'orientation_alpha_sin', 'orientation_alpha_cos', 'drag_distance', 'drag_duration', 'pot_drag_duration', 'approval_swipe_duration', 'approval_swipe_ratio']


In [ ]:
for m in metrics_in_scope:
    plot_participant_average_profiles_stacked(raw_metric_df, m)

In [ ]:
for m in metrics_in_scope:
    plot_session_trajectory_by_device_stacked(raw_metric_df, m)

## Session-level raw summary table

In [ ]:
session_level_summary = (
    filter_time_range(raw_metric_df, TIME_RANGE_S)
    .groupby(['metric', 'participantId', 'sessionId', 'observed_session_number', 'deviceFamily'], dropna=False)
    .agg(session_median=('value', 'median'), session_mean=('value', 'mean'), n_obs=('value', 'size'))
    .reset_index()
    .sort_values(['metric', 'participantId', 'observed_session_number'])
)
session_level_summary

## Notes

Use this notebook to decide whether a raw metric shows:
- **within-session temporal structure** — a trend across the session (fatigue, warm-up,
  task-position effects) rather than a flat, stationary signal
- **between-participant separation through time** — not just pooled shape (that's `02`),
  but whether the *trajectory itself* looks different per person
- **within-participant, across-session consistency or drift** — the actual tool for
  behavioural-drift questions; will only become genuinely informative once more
  participants have multiple sessions

Cross-check anything promising here against `02`'s quantisation table before trusting it
— a consistent trajectory can still be a device artifact rather than a person signal.